In [2]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

In [5]:
import pandas as pd

document_df = pd.read_csv('documents_multihop_v2.csv')
queries_df = pd.read_csv('queries_multihop_v2.csv')

### 벡터스토어 생성 및 문서 업로드

In [6]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
pc = Pinecone()
index_names = pc.list_indexes().names()
print(index_names)

# 압축 문서를 저장할 별도 인덱스 생성
if 'ir-compressed' not in index_names:
    pc.create_index(
        name='ir-multihop',
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(region='us-east-1', cloud='aws')
    )
    print('ir-multihop 인덱스 생성 완료!')
else:
    print('ir-multihop 인덱스가 이미 존재합니다.')

['ir-meta']
ir-multihop 인덱스 생성 완료!


In [10]:
print(queries_df.columns)

Index(['query_id', 'query_text', 'relevant_doc_ids'], dtype='str')


In [12]:
from tqdm.auto import tqdm
from pprint import pprint
from langchain_core.documents import Document 

documents = []


for idx, row in tqdm(document_df.iterrows()): # compressed_df을 행 단위로 순회
    doc_id = row['doc_id']
    query_text = row['content']
    content = row['content']
    doc = Document(
        page_content = content,
        metadata = {'doc_id': doc_id}
    )
    documents.append(doc)

pprint(documents)
   
    

0it [00:00, ?it/s]

[Document(metadata={'doc_id': 'D1'}, page_content='아이폰(iPhone) 스마트폰 시리즈는 애플(Apple Inc.)에 의해 설계 및 마케팅되었습니다.'),
 Document(metadata={'doc_id': 'D2'}, page_content='애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.'),
 Document(metadata={'doc_id': 'D3'}, page_content='쿠퍼티노(Cupertino)는 미국 캘리포니아주에 위치한 도시로, 애플의 본거지로 알려져 있습니다.'),
 Document(metadata={'doc_id': 'D4'}, page_content='팀 쿡(Tim Cook)은 2011년 스티브 잡스의 뒤를 이어 애플(Apple Inc.)의 CEO가 되었습니다.'),
 Document(metadata={'doc_id': 'D5'}, page_content='스티브 잡스(Steve Jobs)는 애플(Apple Inc.)의 공동 창업자였으며 2011년까지 CEO를 역임했습니다.'),
 Document(metadata={'doc_id': 'D6'}, page_content='갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.'),
 Document(metadata={'doc_id': 'D7'}, page_content='삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.'),
 Document(metadata={'doc_id': 'D8'}, page_content='이재용은 현재 삼성전자의 회장직을 맡고 있습니다.'),
 Document(metadata={'doc_id': 'D9'}, page_content='서울은 대한민국의 수도이자 최대 도시로, 한강이 흐르고 있습니다.'),
 Document(metadata={'doc_id': 'D10'}, page_content='삼성전자는 1938년 이병철에 의

In [13]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore


embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small'
)


# ir-meta 인덱스와 연결
vector_store = PineconeVectorStore(
    index_name='ir-meta',
    embedding=embeddings
)

vector_store.add_documents(documents)

['b2d821fd-01f2-4dfa-8810-188b0648bb4e',
 'b66a707a-b315-48e7-9480-e2fd01974a74',
 '2bb47f31-20c8-4a1f-af21-eed1c079e4f4',
 '479809ba-1251-473f-8f64-00dd5446a806',
 'e8ded1e6-55c8-496d-ad62-97b2623341e8',
 '66b4a907-3407-448f-8690-af29ff58df57',
 '2938bd83-a71d-49b5-937d-88c48700f8bc',
 '86eb22cc-a9d6-488d-8414-e216bce85227',
 'f15c8dc5-68c9-4dc0-8115-89ddeca953dc',
 '1ac29bfd-f139-4329-9e97-14f5d46ee928',
 'e78e7732-75d0-49dd-8c1c-4a7a9e3c2aab',
 '9e213562-f643-4712-b8b2-54054486f271',
 '7500f5d2-d6d8-485f-88d6-4c3e958ba785',
 'a25512c1-3935-447a-8042-887db2ea6569',
 '62476064-d5ca-4bfb-aff1-9e68cd156546',
 '0c557414-3c40-42de-b792-a192adaf2208',
 '3793ea40-d362-4b59-ac30-a865f8543362',
 'a0ae3543-5801-4ab2-8c6b-86c210167c34',
 '3d1fe5d3-a57c-46b1-88a6-ea23dfe79211',
 '3158ddea-4a0a-406a-ae9c-af5a49ca4e2e',
 '079262ba-7161-43b8-a117-07a37f62fc79',
 'a00246e9-0d43-493f-a98a-da1a767b15bc',
 '621666f9-1608-4066-bd89-380050328523',
 '962915dc-8105-4a0b-b826-6341e8ab862b',
 '9b6a3d74-b5a7-

In [14]:
vector_store.similarity_search('손흥민', k=3)

[Document(id='2e60d218-922d-4780-98ed-29a401791e39', metadata={'doc_id': 'D29'}, page_content='손흥민은 2021-2022 시즌 프리미어리그에서 아시아 선수 최초로 득점왕(골든 부트)을 수상했습니다.'),
 Document(id='ccba5e6a-d0d2-4e59-beff-29a61af3ae33', metadata={'doc_id': 'D26'}, page_content='손흥민은 잉글랜드 프리미어리그(EPL)의 토트넘 홋스퍼 FC에서 활약하는 대한민국 축구 선수입니다.'),
 Document(id='bea8d216-f607-420c-aa5f-e6fe93b518c6', metadata={'doc_id': 'D30'}, page_content='해리 케인은 토트넘에서 손흥민과 환상적인 호흡을 보여준 잉글랜드 출신 스트라이커입니다.')]

Multihop RAG 구현

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', temperature = 0)
prompt = ChatPromptTemplate.from_template('''
당신은 복잡한 질문에 답하기 위해 정보를 단계적으로 검색하는 AI에이젼트입니다.
사용자의 질문과 현재까지 수집된 정보를 바탕으로,
1. 아직 답을 찾지 못했다면:
  다음에 검색해야 할 가장 구체적이고 필요한 검색어(Query)를 출력하세요
2. 충분한 정보를 찾았다면:
  'ANSWER: '뒤에 최종 정답을 적어서 출력하세요.

### 사용자의 원래질문 ###
{original_question}

### 현재까지 수집된 정보 Context ###
{context}

### 출력지시사항 ###
불필요한 설명없이, '검색어' 또는 'ANSWER: 정답' 형식으로만 출력하세요.

1.추가검색이 필요한 경우, 검색어는 "BTS가 소속된 기획사"인 경우
("검색어" 출력하지 말것)
출력: BTS가 소속된 기획사

2.정답 추론이 가능한 경우
출력: ANSWER: 하이브
''')

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

print(chain.invoke({
    'original_question': '플레이데이터는 어느 역 근처에 있나?',
    'context': ''
}))

플레이데이터 학원 위치 가까운 역



In [16]:
print(chain.invoke({
    'original_question': '플레이데이터는 어느 역 근처에 있나?',
    'context': '플레이데이터는 독산역과 가산디지털단지역 사이에 위치해 있다'
}))

ANSWER: 독산역과 가산디지털단지역 사이 utterance


### 멀티홉 질의응답 함수 구현

In [ ]:
def multihop_search(question, max_hop = 3):
    context = ''
    retrieved_doc_ids = set()

    print(f"사용자의 질문 : {question}")

    for i in range(max_hop):
        print(f"현재 hop 단계는? {i+1}")

        # 에이전트에게 다음 행동(검색어/정답) 질의
        response = chain.invoke({'original_question': question, 'context': context})

        # 정답 도출 여부 확인 : ANSWER로 시작하면 최종 정답
        if response.startswith('ANSWER:'):
            final_answer = response.replace('ANSWER', '').strip()

            print(f"정답은? {final_answer}")
            return final_answer, retrieved_doc_ids

        query = response
        docs = vector_store.similarity_search(query, k = 3)
        for doc in docs:
            retrieved_doc_ids.add(doc.metadata['doc_id'])

        content = 'n'.join([doc.page_content for doc in docs])
        print(f"검색어 : {query}")
        print(f"검색결과 : {content}")

        if context == '':
            context = content
        else:
            context += '\n\n' + content

    print(f"max_hop 내에 정답을 찾지 못하였습니다.")
    return '검색 실패!', retrieved_doc_ids

In [21]:
answer, retrieved_doc_ids = multihop_search('영화 기생충이 상을 받은 영화제는?')
print(answer)
print(retrieved_doc_ids)

사용자의 질문 : 영화 기생충이 상을 받은 영화제는?
현재 hop 단계는? 1
검색어 : 영화 기생충 수상 영화제 주요 수상 내역
검색결과 : 봉준호 감독은 영화 '기생충'으로 칸 영화제에서 최고상인 황금종려상을 수상했습니다.n배우 송강호는 영화 '기생충'에서 기택 역을 맡아 열연했습니다.n'기생충'은 제92회 아카데미 시상식(오스카)에서 작품상, 감독상 등 4관왕을 달성했습니다.
현재 hop 단계는? 2
정답은? : 칸 영화제
: 칸 영화제
{'D24', 'D25', 'D22'}


### 핵심 결과
- 멀티홉 검색은 **단일 검색으로는 해결하기 어려운 복합 질의**에서 효과적으로 동작했다.
- 질문을 단계적으로 분해하고, 각 단계에서 필요한 정보를 순차적으로 수집함으로써  
  **추론 기반 질의에 대한 Recall을 안정적으로 확보**할 수 있었다.

---

### 왜 멀티홉이 필요한가
- 단일 Dense Retrieval은 **한 문서 안에 모든 단서가 존재한다는 가정**에 의존한다.
- 실제 질의는  
  - *“A와 관련된 B는 무엇인가?”*  
  - *“A를 만든 회사의 본사는 어디인가?”*  
  와 같이 **여러 문서에 정보가 분산**된 경우가 많다.
- 멀티홉 검색은 이 문제를  
  **검색 → 추론 → 추가 검색**의 반복 구조로 해결한다.

---

### 실험을 통해 확인된 장점
- 단계별 검색으로 **정답 문서 회수율(Recall) 향상**
- 검색 과정에서 실제로 참조된 문서를 추적 가능
- LLM이 “지금 무엇을 더 찾아야 하는지”를 스스로 판단하여  
  **질의 전개(Query Decomposition)**가 자연스럽게 수행됨

---

### 한계점
- 홉(hop) 수 증가에 따라 **지연 시간과 비용 증가**
- 중간 검색이 잘못되면 이후 단계도 함께 실패할 가능성 존재
- Precision보다는 **Recall 중심 평가에 더 적합**

### 실무 적용 결론
단일 검색으로 해결 가능 → 일반 Retrieval  
복합 추론이 필요한 질문 → Multi-hop Retrieval


- 멀티홉 검색은 **지식 탐색형 QA, 리서치, 에이전트 기반 RAG**에 특히 적합
- 실제 서비스에서는  
  **Self-Query / Metadata → Dense → Multi-hop → ReRank**  
  형태로 결합하는 것이 가장 현실적인 전략이다.

---

### 한 줄 결론 (교안 / 발표용)
> **멀티홉 검색은 분산된 정보를 단계적으로 연결하여, 복합 질의에 대한 추론 가능성을 확장하는 검색 전략이다.**